# Round 4 Manual Challenge: AETHER_CRYSTAL Aggressive EV Optimization

This notebook reflects the updated trading logic for **Round 4: Vanilla Just Isn't Exotic Enough**.

The goal here is not conservative risk control. The goal is to **maximize expected PnL**, while still understanding the drawdown and tail-risk profile. The key update from the earlier pass is that the binary put should be treated as a **buy**, not a sell, under the likely payout assumption.

## Main conclusion

Recommended aggressive portfolio:

| Contract | Side | Volume |
|---|---:|---:|
| AC | No trade | 0 |
| AC_50_P | No trade | 0 |
| AC_50_C | No trade | 0 |
| AC_35_P | No trade | 0 |
| AC_40_P | No trade | 0 |
| AC_45_P | No trade | 0 |
| AC_60_C | Sell | 50 |
| AC_50_P_2 | Buy | 50 |
| AC_50_C_2 | Buy | 50 |
| AC_50_CO | Sell | 50 |
| AC_40_BP | Buy | 50 |
| AC_45_KO | Buy | 500 |

This is an aggressive positive-EV portfolio. It has meaningful negative-tail risk, but it is better aligned with trying to win rather than merely avoid losses.


## Modeling assumptions

- Underlying starts at `S0 = 50`.
- GBM has zero risk-neutral drift.
- Annualized volatility is `251%`.
- 252 trading days per year.
- 4 simulation steps per trading day.
- 2 weeks = 10 trading days = 40 steps.
- 3 weeks = 15 trading days = 60 steps.
- Contract size = 3000.
- Binary put payout is inferred as `20`, because the listed price around 5 is consistent with a 20-unit digital payoff under this volatility regime.
- Knock-out put is modeled as strike 45 with barrier 35. If the exact barrier differs in the UI, update `KO_BARRIER` below and rerun.

The challenge score is based on the **average PnL across 100 simulations**, so this notebook evaluates both single-path PnL and simulated 100-path average score distributions.


In [ ]:
import math
import numpy as np
import pandas as pd

S0 = 50.0
SIGMA = 2.51
R = 0.0

TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY
CONTRACT_SIZE = 3000

N_SIMS = 500_000
N_SCORE_TRIALS = 50_000
SCORE_PATHS = 100
SEED = 20260427

BINARY_PAYOUT = 20.0
KO_BARRIER = 35.0

def weeks_to_years(weeks: float) -> float:
    return (weeks * 5) / TRADING_DAYS_PER_YEAR

def steps_for_weeks(weeks: float) -> int:
    return int(round(weeks * 5 * STEPS_PER_DAY))

T2 = weeks_to_years(2)
T3 = weeks_to_years(3)
STEPS_2W = steps_for_weeks(2)
STEPS_3W = steps_for_weeks(3)

print(T2, T3, STEPS_2W, STEPS_3W)

## Black-Scholes helpers

Vanilla calls and puts can be checked with Black-Scholes. The binary put can also be checked analytically as a cash-or-nothing digital put. The chooser and knock-out option are valued by Monte Carlo because their payoff depends on intermediate path information.

In [ ]:
def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def bs_call(S, K, T, sigma=SIGMA, r=R):
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)

def bs_put(S, K, T, sigma=SIGMA, r=R):
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)

def digital_put(S, K, T, payout=BINARY_PAYOUT, sigma=SIGMA, r=R):
    d2 = (math.log(S / K) + (r - 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    return payout * math.exp(-r * T) * norm_cdf(-d2)

print('2W ATM call:', bs_call(S0, 50, T2))
print('2W ATM put :', bs_put(S0, 50, T2))
print('3W K40 binary put, payout=20:', digital_put(S0, 40, T3))

## Contract table

The table below uses the listed bid/ask and volume limits from the challenge screenshot. Positive order quantity means **buy**. Negative order quantity means **sell**.

In [ ]:
contracts = pd.DataFrame([
    dict(symbol='AC', kind='underlying', weeks=3, strike=np.nan, bid=49.975, ask=50.025, bid_size=200, ask_size=200),
    dict(symbol='AC_50_P', kind='put', weeks=3, strike=50, bid=12.00, ask=12.05, bid_size=50, ask_size=50),
    dict(symbol='AC_50_C', kind='call', weeks=3, strike=50, bid=12.00, ask=12.05, bid_size=50, ask_size=50),
    dict(symbol='AC_35_P', kind='put', weeks=3, strike=35, bid=4.33, ask=4.35, bid_size=50, ask_size=50),
    dict(symbol='AC_40_P', kind='put', weeks=3, strike=40, bid=6.50, ask=6.55, bid_size=50, ask_size=50),
    dict(symbol='AC_45_P', kind='put', weeks=3, strike=45, bid=9.05, ask=9.10, bid_size=50, ask_size=50),
    dict(symbol='AC_60_C', kind='call', weeks=3, strike=60, bid=8.80, ask=8.85, bid_size=50, ask_size=50),
    dict(symbol='AC_50_P_2', kind='put', weeks=2, strike=50, bid=9.70, ask=9.75, bid_size=50, ask_size=50),
    dict(symbol='AC_50_C_2', kind='call', weeks=2, strike=50, bid=9.70, ask=9.75, bid_size=50, ask_size=50),
    dict(symbol='AC_50_CO', kind='chooser', weeks=3, strike=50, bid=22.20, ask=22.30, bid_size=50, ask_size=50),
    dict(symbol='AC_40_BP', kind='binary_put', weeks=3, strike=40, bid=5.00, ask=5.10, bid_size=50, ask_size=50),
    dict(symbol='AC_45_KO', kind='ko_put', weeks=3, strike=45, bid=0.150, ask=0.175, bid_size=500, ask_size=500),
])

contracts

## Simulate AETHER_CRYSTAL paths

The underlying follows discrete-grid GBM with 4 steps per trading day. The knock-out condition is checked only at these simulated grid points, matching the challenge statement.

In [ ]:
rng = np.random.default_rng(SEED)
dt = 1 / STEPS_PER_YEAR

z = rng.standard_normal((N_SIMS, STEPS_3W))
log_paths = math.log(S0) + np.cumsum((-0.5 * SIGMA**2) * dt + SIGMA * math.sqrt(dt) * z, axis=1)
paths = np.exp(log_paths)

S_2W = paths[:, STEPS_2W - 1]
S_3W = paths[:, STEPS_3W - 1]
path_min_3W = paths.min(axis=1)

pd.Series(S_3W).describe(percentiles=[0.001, 0.005, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995, 0.999])

## Payoff functions

For a buy, PnL is `payoff - ask`. For a sell, PnL is `bid - payoff`. Both are multiplied by volume and contract size.

In [ ]:
def mc_payoff(symbol):
    row = contracts.loc[contracts.symbol == symbol].iloc[0]
    K = row.strike

    if row.kind == 'underlying':
        return S_3W
    if row.kind == 'call':
        ST = S_2W if row.weeks == 2 else S_3W
        return np.maximum(ST - K, 0.0)
    if row.kind == 'put':
        ST = S_2W if row.weeks == 2 else S_3W
        return np.maximum(K - ST, 0.0)
    if row.kind == 'chooser':
        # At 2 weeks, choose call if S_2W >= K, otherwise choose put.
        return np.where(S_2W >= K, np.maximum(S_3W - K, 0.0), np.maximum(K - S_3W, 0.0))
    if row.kind == 'binary_put':
        return BINARY_PAYOUT * (S_3W < K)
    if row.kind == 'ko_put':
        alive = path_min_3W > KO_BARRIER
        return np.where(alive, np.maximum(K - S_3W, 0.0), 0.0)
    raise ValueError(f'Unknown kind: {row.kind}')

payoffs = {sym: mc_payoff(sym) for sym in contracts.symbol}

def analytical_value(row):
    if row.kind == 'underlying':
        return S0
    T = weeks_to_years(row.weeks)
    if row.kind == 'call':
        return bs_call(S0, row.strike, T)
    if row.kind == 'put':
        return bs_put(S0, row.strike, T)
    if row.kind == 'binary_put':
        return digital_put(S0, row.strike, T)
    return np.nan

def position_pnl(symbol, qty):
    if qty == 0:
        return np.zeros(N_SIMS)
    row = contracts.loc[contracts.symbol == symbol].iloc[0]
    payoff = payoffs[symbol]
    if qty > 0:
        return qty * CONTRACT_SIZE * (payoff - row.ask)
    return (-qty) * CONTRACT_SIZE * (row.bid - payoff)

## Valuation and edge table

This table shows the fair value, buy EV, sell EV, and raw best side for every contract. The major correction is visible on `AC_40_BP`: using a 20-unit binary payoff, the fair value is above the ask, so the aggressive expected-value trade is **buy**, not sell.

In [ ]:
rows = []
for _, row in contracts.iterrows():
    fair_mc = float(np.mean(payoffs[row.symbol]))
    fair_bs = analytical_value(row)
    buy_ev = fair_mc - row.ask
    sell_ev = row.bid - fair_mc
    if buy_ev > sell_ev and buy_ev > 0:
        best_side = 'BUY'
        best_volume = int(row.ask_size)
        best_ev_unit = buy_ev
    elif sell_ev > 0:
        best_side = 'SELL'
        best_volume = int(row.bid_size)
        best_ev_unit = sell_ev
    else:
        best_side = 'NO TRADE'
        best_volume = 0
        best_ev_unit = 0.0

    rows.append({
        'symbol': row.symbol,
        'kind': row.kind,
        'bid': row.bid,
        'ask': row.ask,
        'fair_mc': fair_mc,
        'fair_bs_if_available': fair_bs,
        'buy_ev_per_unit': buy_ev,
        'sell_ev_per_unit': sell_ev,
        'raw_best_side': best_side,
        'raw_best_volume': best_volume,
        'raw_best_ev_per_unit': best_ev_unit,
        'raw_best_expected_pnl': best_ev_unit * best_volume * CONTRACT_SIZE,
    })

valuation = pd.DataFrame(rows).sort_values('raw_best_expected_pnl', ascending=False)
valuation

## Final aggressive order set

This is the order set I would submit if the objective is to maximize profit and accept real risk. It is more aggressive than the first risk-aware version because it takes the binary put long and sells the 60 call.

In [ ]:
# Positive = buy, negative = sell.
aggressive_orders = {
    'AC': 0,
    'AC_50_P': 0,
    'AC_50_C': 0,
    'AC_35_P': 0,
    'AC_40_P': 0,
    'AC_45_P': 0,
    'AC_60_C': -50,
    'AC_50_P_2': 50,
    'AC_50_C_2': 50,
    'AC_50_CO': -50,
    'AC_40_BP': 50,
    'AC_45_KO': 500,
}

order_sheet = []
for sym, qty in aggressive_orders.items():
    order_sheet.append({
        'symbol': sym,
        'side': 'BUY' if qty > 0 else ('SELL' if qty < 0 else 'NO TRADE'),
        'volume': abs(qty),
        'signed_qty': qty,
    })

order_sheet = pd.DataFrame(order_sheet)
order_sheet

## Single-path PnL distribution

This is the distribution of PnL if the portfolio were evaluated on one underlying path. The challenge uses the average of 100 paths, so this is harsher than the actual scoring distribution, but it is useful for understanding tail risk.

In [ ]:
portfolio_pnl = sum(position_pnl(sym, qty) for sym, qty in aggressive_orders.items())

single_path_summary = pd.Series(portfolio_pnl).describe(
    percentiles=[0.001, 0.005, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995, 0.999]
).to_frame('single_path_pnl')

single_path_extra = pd.DataFrame({
    'single_path_pnl': {
        'prob_negative': float(np.mean(portfolio_pnl < 0)),
        'cvar_5pct': float(portfolio_pnl[portfolio_pnl <= np.quantile(portfolio_pnl, 0.05)].mean()),
        'cvar_1pct': float(portfolio_pnl[portfolio_pnl <= np.quantile(portfolio_pnl, 0.01)].mean()),
    }
})

pd.concat([single_path_summary, single_path_extra])

## Position-level contribution

This breaks down each leg's expected contribution and risk. The short 60 call and short chooser create tail risk, while the 2-week straddle, binary put, and knock-out put drive expected upside under this model.

In [ ]:
position_rows = []
for sym, qty in aggressive_orders.items():
    pnl = position_pnl(sym, qty)
    position_rows.append({
        'symbol': sym,
        'qty': qty,
        'side': 'BUY' if qty > 0 else ('SELL' if qty < 0 else 'NO TRADE'),
        'volume': abs(qty),
        'expected_pnl': float(np.mean(pnl)),
        'std_pnl': float(np.std(pnl)),
        'p05_pnl': float(np.quantile(pnl, 0.05)),
        'p01_pnl': float(np.quantile(pnl, 0.01)),
        'min_sim_pnl': float(np.min(pnl)),
        'max_sim_pnl': float(np.max(pnl)),
    })

position_summary = pd.DataFrame(position_rows).sort_values('expected_pnl', ascending=False)
position_summary

## 100-path average score simulation

The actual challenge score is the **average PnL across 100 simulations**. The next cell bootstraps from the simulated path-level PnL distribution to approximate the score distribution.

In [ ]:
score_rng = np.random.default_rng(SEED + 1)
idx = score_rng.integers(0, N_SIMS, size=(N_SCORE_TRIALS, SCORE_PATHS))
score_samples = portfolio_pnl[idx].mean(axis=1)

score_summary = pd.Series(score_samples).describe(
    percentiles=[0.001, 0.005, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995, 0.999]
).to_frame('avg_100_path_score')

score_extra = pd.DataFrame({
    'avg_100_path_score': {
        'prob_negative': float(np.mean(score_samples < 0)),
        'cvar_5pct': float(score_samples[score_samples <= np.quantile(score_samples, 0.05)].mean()),
        'cvar_1pct': float(score_samples[score_samples <= np.quantile(score_samples, 0.01)].mean()),
    }
})

pd.concat([score_summary, score_extra])

## Compare against the earlier risk-aware version

The earlier balanced version omitted the binary put and 60 call. This comparison makes the tradeoff explicit: the aggressive version has higher expected PnL, but worse tails.

In [ ]:
balanced_orders = {
    'AC': 0,
    'AC_50_P': 0,
    'AC_50_C': 0,
    'AC_35_P': 0,
    'AC_40_P': 0,
    'AC_45_P': 0,
    'AC_60_C': 0,
    'AC_50_P_2': 50,
    'AC_50_C_2': 50,
    'AC_50_CO': -50,
    'AC_40_BP': 0,
    'AC_45_KO': 500,
}

balanced_pnl = sum(position_pnl(sym, qty) for sym, qty in balanced_orders.items())

def summarize_pnl(name, pnl):
    return {
        'portfolio': name,
        'mean': float(np.mean(pnl)),
        'median': float(np.median(pnl)),
        'std': float(np.std(pnl)),
        'p05': float(np.quantile(pnl, 0.05)),
        'p01': float(np.quantile(pnl, 0.01)),
        'min': float(np.min(pnl)),
        'prob_negative': float(np.mean(pnl < 0)),
        'sharpe_like': float(np.mean(pnl) / np.std(pnl)),
    }

comparison_single_path = pd.DataFrame([
    summarize_pnl('balanced', balanced_pnl),
    summarize_pnl('aggressive', portfolio_pnl),
])

comparison_single_path

## Final submit block

Use this in the manual challenge UI.

In [ ]:
for sym, qty in aggressive_orders.items():
    side = 'BUY' if qty > 0 else ('SELL' if qty < 0 else 'NO TRADE')
    print(f'{sym:12s} {side:8s} {abs(qty)}')